# 06. Sorting, Reindexing & Reshaping: Beginner Guide

### 🌟 What is Sorting, Reindexing & Reshaping in Pandas?
Restructuring tables is essential when preparing data for reports or machine learning. This notebook covers sorting values and indices, realigning schemas with `.reindex()`, pivoting data with `pivot_table()`, and flattening multi-column structures with `melt()`.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Sorting**: Covers `.sort_values()` and `.sort_index()`.
- **Reindexing**: Covers `.reindex()`, `.reset_index()`, and `.set_index()`.
- **Reshaping & Pivoting**: Covers `pd.melt()`, `df.pivot()`, `df.pivot_table()`, `.stack()`, and `.unstack()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount   card_type  \
0       TX110686      C82845       M2697             1216.33        Visa   
1       TX107170      C85674       M3868              324.99  MasterCard   

  transaction_status device_type  account_age_months transaction_date region  \
0             Failed         POS                  56       07-06-2025  North   
1           Reversed     Desktop                 112       10/05/2025   East   

   is_fraud  
0         0  
1         0  


### 🔹 Sorting Values with `.sort_values()`
Sorts transactions by amount in descending order. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.sort_values(by='transaction_amount', ascending=False)`


In [2]:
sorted_tx = df.sort_values(by='transaction_amount', ascending=False)
print('Top 3 Largest Transactions:\n', sorted_tx[['transaction_id', 'transaction_amount', 'card_type', 'is_fraud']].head(3))

Top 3 Largest Transactions:
       transaction_id  transaction_amount   card_type  is_fraud
12028       TX113019             1999.98  MasterCard         0
5054        TX101702             1999.85        Amex         1
7337        TX108046             1999.74  MasterCard         1


### 🔹 Sorting by Index with `.sort_index()`
Sorts DataFrame chronologically by date index. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.set_index('transaction_date').sort_index()`


In [3]:
ts_df = df.set_index('transaction_date')
print('Sorted by Index Head:\n', ts_df.head(2))

Sorted by Index Head:
                  transaction_id customer_id merchant_id  transaction_amount  \
transaction_date                                                              
07-06-2025             TX110686      C82845       M2697             1216.33   
10/05/2025             TX107170      C85674       M3868              324.99   

                   card_type transaction_status device_type  \
transaction_date                                              
07-06-2025              Visa             Failed         POS   
10/05/2025        MasterCard           Reversed     Desktop   

                  account_age_months region  is_fraud  
transaction_date                                       
07-06-2025                        56  North         0  
10/05/2025                       112   East         0  


### 🔹 Conforming to New Labels: `.reindex()`
Reindexes DataFrame with a custom index range. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.reindex(new_index, fill_value=0)`


In [4]:
reindexed_tx = df.head(5).reindex([0, 1, 2, 999], fill_value=0.0)
print('Reindexed Transaction Slice:\n', reindexed_tx[['transaction_id', 'transaction_amount']])

Reindexed Transaction Slice:
     transaction_id  transaction_amount
0         TX110686             1216.33
1         TX107170              324.99
2         TX108328              136.66
999            0.0                0.00


### 🔹 Resetting Index: `.reset_index()`
Restores default 0-indexed integer RangeIndex. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Sets only store unique elements and provide $O(1)$ instant lookup time, making `item in my_set` extremely fast.

**Syntax:** `sorted_tx.reset_index(drop=True)`


In [5]:
print('Reset Index Head:\n', sorted_tx.reset_index(drop=True)[['transaction_id', 'transaction_amount']].head(3))

Reset Index Head:
   transaction_id  transaction_amount
0       TX113019             1999.98
1       TX101702             1999.85
2       TX108046             1999.74


### 🔹 Setting Index: `.set_index()`
Designates `transaction_id` as primary index. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Sets only store unique elements and provide $O(1)$ instant lookup time, making `item in my_set` extremely fast.

**Syntax:** `df.set_index('transaction_id')`


In [6]:
tx_indexed = df.set_index('transaction_id')
print('Transaction ID Indexed Head:\n', tx_indexed.head(2))

Transaction ID Indexed Head:
                customer_id merchant_id  transaction_amount   card_type  \
transaction_id                                                           
TX110686            C82845       M2697             1216.33        Visa   
TX107170            C85674       M3868              324.99  MasterCard   

               transaction_status device_type  account_age_months  \
transaction_id                                                      
TX110686                   Failed         POS                  56   
TX107170                 Reversed     Desktop                 112   

               transaction_date region  is_fraud  
transaction_id                                    
TX110686             07-06-2025  North         0  
TX107170             10/05/2025   East         0  


### 🔹 Unpivoting Wide to Long: `pd.melt()`
Unpivots metric columns into long format key-value pairs. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `pd.melt(df, id_vars=['transaction_id'], value_vars=['transaction_amount', 'account_age_months'])`


In [7]:
melted_tx = pd.melt(df.head(5), id_vars=['transaction_id'], value_vars=['transaction_amount', 'account_age_months'], var_name='metric', value_name='metric_value')
print('Melted Transaction Metrics:\n', melted_tx)

Melted Transaction Metrics:
   transaction_id              metric  metric_value
0       TX110686  transaction_amount       1216.33
1       TX107170  transaction_amount        324.99
2       TX108328  transaction_amount        136.66
3       TX108563  transaction_amount        124.21
4       TX107002  transaction_amount       1284.68
5       TX110686  account_age_months         56.00
6       TX107170  account_age_months        112.00
7       TX108328  account_age_months         68.00
8       TX108563  account_age_months         50.00
9       TX107002  account_age_months         96.00


### 🔹 Long to Wide Pivoting: `df.pivot()`
Pivots unique key combinations into wide format. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.pivot(index='customer_id', columns='card_type', values='amount')`


In [8]:
print('df.pivot Syntax: df.pivot(index="customer_id", columns="card_type", values="amount")')

df.pivot Syntax: df.pivot(index="customer_id", columns="card_type", values="amount")


### 🔹 Pivot Tables: `df.pivot_table()`
Constructs multidimensional regional card spending matrices with margins. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.pivot_table(index='region', columns='card_type', values='transaction_amount', aggfunc='mean', margins=True)`


In [9]:
piv_spending = df.pivot_table(index='region', columns='card_type', values='transaction_amount', aggfunc='mean', margins=True)
print('Regional Card Spending Matrix (Mean Amount):\n', piv_spending.round(2))

Regional Card Spending Matrix (Mean Amount):
 card_type     Amex  Discover  MasterCard     Visa      All
region                                                    
 East      1003.79    810.77     1016.64   905.95   930.41
 North     1167.63   1234.21     1126.08   894.14  1103.02
 South      965.99   1052.49     1192.08  1006.45  1045.99
 West       925.65    976.08     1037.67   742.78   908.58
East       1011.58   1007.47      981.91   982.98   995.98
North      1016.36   1024.76     1021.45  1007.94  1017.58
South       977.65   1009.52     1024.76   999.27  1002.57
West       1021.24    990.75      980.81  1000.86   998.32
east       1063.68   1180.74     1336.47  1014.07  1149.98
north      1102.00   1162.32      931.94  1195.61  1091.41
south      1145.76   1058.21     1254.93  1130.91  1150.24
west       1274.07   1272.33      906.03   953.86  1113.05
All        1009.23   1012.05     1006.28   996.92  1006.14


### 🔹 Stacking Columns to Rows: `.stack()`
Stacks pivoted card columns into hierarchical multi-level row index. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `piv_spending.stack()`


In [10]:
stacked_piv = piv_spending.stack()
print('Stacked Pivot Series Head:\n', stacked_piv.head())

Stacked Pivot Series Head:
 region  card_type 
East    Amex          1003.787647
        Discover       810.773158
        MasterCard    1016.640588
        Visa           905.949444
        All            930.409296
dtype: float64


### 🔹 Unstacking Rows to Columns: `.unstack()`
Unstacks hierarchical row levels back to column headers. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `stacked_piv.unstack()`


In [11]:
print('Unstacked DataFrame:\n', stacked_piv.unstack().head(2))

Unstacked DataFrame:
 card_type         Amex     Discover   MasterCard        Visa          All
region                                                                   
East       1003.787647   810.773158  1016.640588  905.949444   930.409296
North      1167.632778  1234.214783  1126.083333  894.138636  1103.023210


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Regional Fraud Risk Contingency Pivot Table

**Approach:** Generate a pivot contingency matrix showing fraud rates across regions and device types.
**Syntax:** `df.pivot_table(index='region', columns='device_type', values='is_fraud', aggfunc='mean') * 100`


In [12]:
fraud_matrix = df.pivot_table(index='region', columns='device_type', values='is_fraud', aggfunc='mean') * 100
print('Regional Device Fraud Rate Matrix (%):\n', fraud_matrix.round(2))

Regional Device Fraud Rate Matrix (%):
 device_type    ATM  Desktop  Mobile    POS
region                                    
 East         4.35     4.00    0.00  15.38
 North        4.76     5.00   23.53   8.33
 South        0.00    20.00   15.38   4.17
 West         5.88     0.00   10.00  15.00
East          8.32    10.38   10.98   9.78
North        11.21    10.70   11.74  11.43
South        11.62    10.43   10.19   9.73
West         12.96     9.02    9.12  11.10
east         19.05    23.08   13.04  13.64
north        11.76    12.00   22.22  28.57
south        12.00     8.70    6.67  16.67
west         12.50    22.22    0.00   5.88
